<a href="https://colab.research.google.com/github/ahmedsalha130/CMS/blob/main/Data_Cleaing_Main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install requests pandas openpyxl

In [ ]:
import requests
import pandas as pd
import re
from collections import Counter
from datetime import datetime
from openpyxl.styles import Font # أضفنا هذه لاستخدام التنسيق الاحترافي

# =========================
# 1) SETTINGS & FILTERS
# =========================

KOBO_TOKEN = "e99118b88e6edfdb76fa4fec51c1345e5e9b714a"
ASSET_UID = "adHkBRHo7KXvszurjKeu4j"

# الفلاتر
START_DATE = "2026-06-01"
END_DATE = "2026-06-24"
HOSPITAL_FILTER = "Alqsa_h"  # مثال: "AR_h",Ns_h,Alqsa_h,IN_h,Alshifa_h,RA_h

CURRENT_YEAR = datetime.now().year

# --- تعديل 1: تسمية الملف بشكل ديناميكي ---
hosp_name = HOSPITAL_FILTER if HOSPITAL_FILTER else "All_Hospitals"
start_str = START_DATE if START_DATE else "Start"
end_str = END_DATE if END_DATE else "End"
OUTPUT_FILE = f"Data_Quality_{hosp_name}_{start_str}_to_{end_str}.xlsx"

DATA_URL = f"https://kf.kobotoolbox.org/api/v2/assets/{ASSET_UID}/data/?format=json&limit=1000"

headers = {
    "Authorization": f"Token {KOBO_TOKEN}"
}

# =========================
# 2) KOBO COLUMNS MAPPING
# =========================

COL_ID = "main/patient_section/patient_id"
COL_NAME = "main/patient_section/patient_new/new_patient_name"
COL_BIRTH_YEAR = "main/patient_section/patient_new/_Date_of_Birth"
COL_GENDER = "main/patient_section/patient_new/new_patient_gender"
COL_VISIT_DATE = "main/patient_section/patient_new/_Date_of_visit"
COL_HOSPITAL = "main/visit_data/hospital"
COL_SUBMISSION_TIME = "_submission_time"

# =========================
# 3) FUNCTIONS
# =========================

def clean_name(value):
    if pd.isna(value):
        return ""
    value = str(value).strip()
    value = re.sub(r"\s+", " ", value)
    value = value.replace("ـ", "")
    replacements = {
        "أ": "ا", "إ": "ا", "آ": "ا",
        "ى": "ي", "ة": "ه",
        "ؤ": "و", "ئ": "ي",
    }
    for old, new in replacements.items():
        value = value.replace(old, new)
    value = re.sub(r"[^\u0600-\u06FFa-zA-Z0-9\s]", "", value)
    return value.strip()

def clean_id(value):
    if pd.isna(value):
        return ""
    value = str(value).strip()
    value = value.replace(".0", "")
    value = re.sub(r"\D", "", value)
    return value

def clean_gender(value):
    if pd.isna(value):
        return ""
    value = str(value).strip().lower()
    male_values = ["male", "m", "ذكر", "رجل", "boy", "man"]
    female_values = ["female", "f", "انثى", "أنثى", "امرأة", "امراة", "girl", "woman"]
    if value in male_values:
        return "Male"
    if value in female_values:
        return "Female"
    return value

def clean_birth_year(value):
    if pd.isna(value):
        return pd.NA
    value = str(value).strip().replace(".0", "")
    match = re.search(r"\d{4}", value)
    if not match:
        return pd.NA
    year = int(match.group())
    if year < 1900 or year > CURRENT_YEAR:
        return pd.NA
    return year

def most_common(series):
    values = [x for x in series if pd.notna(x) and str(x).strip() != ""]
    if not values:
        return ""
    return Counter(values).most_common(1)[0][0]

def remove_timezone(df):
    for col in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[col]):
            try:
                df[col] = df[col].dt.tz_localize(None)
            except Exception:
                pass
    return df

# =========================
# 4) PULL FROM KOBO API
# =========================

all_records = []
url = DATA_URL

print("جاري جلب البيانات من KoboToolbox...")
while url:
    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        print(response.text)
        raise SystemExit("API Error")
    data = response.json()
    all_records.extend(data.get("results", []))
    url = data.get("next")

df = pd.json_normalize(all_records)
print("إجمالي السجلات المسحوبة:", len(df))

# =========================
# 5) FILTER DATE & HOSPITAL
# =========================

if COL_VISIT_DATE in df.columns:
    df[COL_VISIT_DATE] = pd.to_datetime(df[COL_VISIT_DATE], errors="coerce").dt.date

if START_DATE:
    start = pd.to_datetime(START_DATE).date()
    df = df[df[COL_VISIT_DATE] >= start]

if END_DATE:
    end = pd.to_datetime(END_DATE).date()
    df = df[df[COL_VISIT_DATE] <= end]

if HOSPITAL_FILTER and COL_HOSPITAL in df.columns:
    df = df[df[COL_HOSPITAL] == HOSPITAL_FILTER]

df = df.copy()
print("إجمالي السجلات بعد الفلترة:", len(df))

if len(df) == 0:
    raise SystemExit("لا توجد بيانات تطابق الفلاتر المدخلة.")

df["source_row"] = range(1, len(df) + 1)

# =========================
# 6) STANDARDIZATION
# =========================

df["clean_id"] = df[COL_ID].apply(clean_id)
df["clean_name"] = df[COL_NAME].apply(clean_name)
df["clean_gender"] = df[COL_GENDER].apply(clean_gender)
df["clean_birth_year"] = df[COL_BIRTH_YEAR].apply(clean_birth_year)

# =========================
# 7) MASTER IDENTITY (CONFLICTS)
# =========================

valid_id_df = df[df["clean_id"] != ""].copy()

master = (
    valid_id_df
    .groupby("clean_id")
    .agg(
        visits_count=("clean_id", "count"),
        master_name=("clean_name", most_common),
        master_gender=("clean_gender", most_common),
        master_birth_year=("clean_birth_year", most_common),
        unique_names=("clean_name", lambda x: sorted(set([v for v in x if v != ""]))),
        unique_genders=("clean_gender", lambda x: sorted(set([v for v in x if v != ""]))),
        unique_birth_years=("clean_birth_year", lambda x: sorted(set([v for v in x if pd.notna(v)]))),
    )
    .reset_index()
)

master["problem_same_id_different_name"] = master["unique_names"].apply(lambda x: len(x) > 1)
master["problem_same_id_different_gender"] = master["unique_genders"].apply(lambda x: len(x) > 1)
master["problem_same_id_different_birth_year"] = master["unique_birth_years"].apply(lambda x: len(x) > 1)

master["has_identity_problem"] = (
    master["problem_same_id_different_name"] |
    master["problem_same_id_different_gender"] |
    master["problem_same_id_different_birth_year"]
)

df = df.merge(
    master[
        [
            "clean_id", "visits_count", "master_name", "master_gender", "master_birth_year",
            "problem_same_id_different_name", "problem_same_id_different_gender",
            "problem_same_id_different_birth_year", "has_identity_problem"
        ]
    ],
    on="clean_id",
    how="left"
)

# =========================
# 8) BASIC PROBLEMS
# =========================

df["problem_missing_id"] = df["clean_id"].eq("")
df["problem_missing_name"] = df["clean_name"].eq("")
df["problem_missing_gender"] = df["clean_gender"].eq("")
df["problem_missing_or_invalid_birth_year"] = df["clean_birth_year"].isna()

df["problem_id_too_short"] = df["clean_id"].apply(lambda x: x != "" and len(x) < 6)

df["problem_fake_id"] = df["clean_id"].isin([
    "0", "00", "000", "0000", "00000", "000000", "000000000",
    "111111111", "123456789", "999999999", "123456", "123123"
])

# =========================
# 9) FLAGS & RISK LEVELS
# =========================

problem_columns = [
    "problem_missing_id",
    "problem_missing_name",
    "problem_missing_gender",
    "problem_missing_or_invalid_birth_year",
    "problem_id_too_short",
    "problem_fake_id",
    "problem_same_id_different_name",
    "problem_same_id_different_gender",
    "problem_same_id_different_birth_year",
]

labels = {
    "problem_missing_id": "MISSING_ID",
    "problem_missing_name": "MISSING_NAME",
    "problem_missing_gender": "MISSING_GENDER",
    "problem_missing_or_invalid_birth_year": "MISSING_OR_INVALID_BIRTH_YEAR",
    "problem_id_too_short": "ID_TOO_SHORT",
    "problem_fake_id": "FAKE_OR_TEST_ID",
    "problem_same_id_different_name": "SAME_ID_DIFFERENT_NAME",
    "problem_same_id_different_gender": "SAME_ID_DIFFERENT_GENDER",
    "problem_same_id_different_birth_year": "SAME_ID_DIFFERENT_BIRTH_YEAR",
}

def build_flags(row):
    return " | ".join([labels[col] for col in problem_columns if row.get(col) is True])

df["problem_flags"] = df.apply(build_flags, axis=1)

def risk_level(flags):
    if flags == "":
        return "Clean"

    critical_flags = [
        "MISSING_ID",
        "FAKE_OR_TEST_ID",
        "SAME_ID_DIFFERENT_GENDER",
        "SAME_ID_DIFFERENT_BIRTH_YEAR",
    ]

    if any(flag in flags for flag in critical_flags):
        return "Critical"
    return "Warning"

df["risk_level"] = df["problem_flags"].apply(risk_level)

# =========================
# 10) ISSUE SHEETS PREP
# =========================

base_cols = [
    "_id", "source_row", COL_VISIT_DATE, COL_ID, COL_NAME, COL_GENDER, COL_BIRTH_YEAR,
    "clean_id", "clean_name", "clean_gender", "clean_birth_year",
    "master_name", "master_gender", "master_birth_year",
    "problem_flags", "risk_level"
]

base_cols = [c for c in base_cols if c in df.columns]

def issue_sheet(problem_col, issue_name, explanation):
    temp = df[df[problem_col].fillna(False)].copy()
    temp.insert(0, "Issue_Name", issue_name)
    temp.insert(1, "Explanation_Arabic", explanation)
    cols = ["Issue_Name", "Explanation_Arabic"] + base_cols
    cols = [c for c in cols if c in temp.columns]
    return temp[cols]

sheet_missing_id = issue_sheet("problem_missing_id", "MISSING_ID", "رقم هوية المريض مفقود أو فارغ.")
sheet_missing_name = issue_sheet("problem_missing_name", "MISSING_NAME", "اسم المريض مفقود أو فارغ.")
sheet_missing_gender = issue_sheet("problem_missing_gender", "MISSING_GENDER", "جنس المريض مفقود أو غير محدد.")
sheet_missing_birth_year = issue_sheet("problem_missing_or_invalid_birth_year", "MISSING_OR_INVALID_BIRTH_YEAR", "سنة الميلاد مفقودة أو خارج النطاق المنطقي.")
sheet_id_too_short = issue_sheet("problem_id_too_short", "ID_TOO_SHORT", "رقم الهوية المدخل قصير جداً (أقل من 6 أرقام).")
sheet_fake_id = issue_sheet("problem_fake_id", "FAKE_OR_TEST_ID", "رقم الهوية المدخل يبدو وهمياً أو مخصصاً للاختبار.")
sheet_same_id_name = issue_sheet("problem_same_id_different_name", "SAME_ID_DIFFERENT_NAME", "رقم الهوية مرتبط بأكثر من اسم مختلف.")
sheet_same_id_gender = issue_sheet("problem_same_id_different_gender", "SAME_ID_DIFFERENT_GENDER", "رقم الهوية مرتبط بأكثر من جنس مختلف.")
sheet_same_id_birth_year = issue_sheet("problem_same_id_different_birth_year", "SAME_ID_DIFFERENT_BIRTH_YEAR", "رقم الهوية مرتبط بأكثر من سنة ميلاد مختلفة.")

# =========================
# 11) SUMMARY
# =========================

# --- تعديل 2: إضافة معلومات المستشفى والتاريخ لملخص البيانات ---
summary_rows = [
    ["Hospital / Facility", hosp_name],
    ["Period Start", start_str],
    ["Period End", end_str],
    ["-------------------", "-------------------"],
    ["Total records pulled", len(df)],
    ["Unique Patient IDs", df["clean_id"].nunique()],
    ["Clean records", int((df["risk_level"] == "Clean").sum())],
    ["Warning records", int((df["risk_level"] == "Warning").sum())],
    ["Critical records", int((df["risk_level"] == "Critical").sum())],
    ["-------------------", "-------------------"],
]

for col in problem_columns:
    summary_rows.append([labels[col], int(df[col].fillna(False).sum())])

summary = pd.DataFrame(summary_rows, columns=["Metric", "Value/Count"])

identity_summary = master[master["has_identity_problem"]].copy()
records_with_problems = df[df["problem_flags"] != ""].copy()

# =========================
# 12) EXPORT TO EXCEL & FORMATTING
# =========================

tables = [
    df, records_with_problems, identity_summary,
    sheet_missing_id, sheet_missing_name, sheet_missing_gender,
    sheet_missing_birth_year, sheet_id_too_short, sheet_fake_id,
    sheet_same_id_name, sheet_same_id_gender, sheet_same_id_birth_year
]

for i, table in enumerate(tables):
    tables[i] = remove_timezone(table)

print(f"جاري حفظ وتنسيق الملف: {OUTPUT_FILE} ...")

with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    summary.to_excel(writer, sheet_name="00_Summary", index=False)
    identity_summary.to_excel(writer, sheet_name="01_ID_Conflicts", index=False)
    sheet_missing_id.to_excel(writer, sheet_name="02_Missing_ID", index=False)
    sheet_missing_name.to_excel(writer, sheet_name="03_Missing_Name", index=False)
    sheet_missing_gender.to_excel(writer, sheet_name="04_Missing_Gender", index=False)
    sheet_missing_birth_year.to_excel(writer, sheet_name="05_BirthYear_Issues", index=False)
    sheet_id_too_short.to_excel(writer, sheet_name="06_ID_Too_Short", index=False)
    sheet_fake_id.to_excel(writer, sheet_name="07_Fake_ID", index=False)
    sheet_same_id_name.to_excel(writer, sheet_name="08_Same_ID_Name", index=False)
    sheet_same_id_gender.to_excel(writer, sheet_name="09_Same_ID_Gender", index=False)
    sheet_same_id_birth_year.to_excel(writer, sheet_name="10_Same_ID_BirthYear", index=False)
    records_with_problems.to_excel(writer, sheet_name="11_All_Problems", index=False)
    df.to_excel(writer, sheet_name="12_All_Checked", index=False)

    # --- تعديل 3: تنسيق احترافي لجميع الصفحات ---
    for sheetname in writer.sheets:
        worksheet = writer.sheets[sheetname]

        # تضخيم خط الترويسة (Header)
        for cell in worksheet["1:1"]:
            cell.font = Font(bold=True)

        # ضبط عرض الأعمدة تلقائياً بناءً على حجم المحتوى
        for col in worksheet.columns:
            max_length = 0
            column = col[0].column_letter # جلب حرف العمود (مثال: A, B, C)
            for cell in col:
                try:
                    if len(str(cell.value)) > max_length:
                        max_length = len(str(cell.value))
                except:
                    pass

            adjusted_width = (max_length + 2)
            # وضع حد أقصى للعرض حتى لا تصبح الأعمدة التي تحتوي على نصوص طويلة عريضة بشكل مزعج
            worksheet.column_dimensions[column].width = min(adjusted_width, 45)

print("✅ تمت عملية التدقيق بنجاح!")
print(f"تم حفظ الملف باسم: {OUTPUT_FILE}")
print("\n--- الإحصائيات العامة ---")
print(summary.to_string(index=False))

جاري جلب البيانات من KoboToolbox...
إجمالي السجلات المسحوبة: 134387
إجمالي السجلات بعد الفلترة: 3615
جاري حفظ وتنسيق الملف: Data_Quality_Alqsa_h_2026-06-01_to_2026-06-24.xlsx ...
✅ تمت عملية التدقيق بنجاح!
تم حفظ الملف باسم: Data_Quality_Alqsa_h_2026-06-01_to_2026-06-24.xlsx

--- الإحصائيات العامة ---
                       Metric         Value/Count
          Hospital / Facility             Alqsa_h
                 Period Start          2026-06-01
                   Period End          2026-06-24
          ------------------- -------------------
         Total records pulled                3615
           Unique Patient IDs                2878
                Clean records                2647
              Warning records                 688
             Critical records                 280
          ------------------- -------------------
                   MISSING_ID                   0
                 MISSING_NAME                   0
               MISSING_GENDER                  

In [ ]:
import requests
import pandas as pd
import re
from collections import Counter
from datetime import datetime

# =========================
# 1) SETTINGS & FILTERS
# =========================

KOBO_TOKEN = "e99118b88e6edfdb76fa4fec51c1345e5e9b714a"
ASSET_UID = "a6xjDqv8kx2YeaNZBetfi8" # UID الخاص بنموذج العيون

# الفلاتر (اترك القيمة "" أو None لجلب كافة البيانات)
START_DATE = "2026-06-01"
END_DATE = "2026-06-20"
CLINIC_FILTER = "" # مثال: "c1_outpatient" أو "c5_outpatient" ، اتركها فارغة لجلب الكل

CURRENT_YEAR = datetime.now().year
OUTPUT_FILE = "Eye_Clinic_Data_Quality_Audit.xlsx"

DATA_URL = f"https://kf.kobotoolbox.org/api/v2/assets/{ASSET_UID}/data/?format=json&limit=1000"

headers = {
    "Authorization": f"Token {KOBO_TOKEN}"
}

# =========================
# 2) KOBO COLUMNS MAPPING
# =========================

COL_ID = "patient/id_number"
COL_NAME = "patient/patient_name"
COL_BIRTH_YEAR = "patient/_dob"
COL_GENDER = "patient/patient_gender"
COL_VISIT_DATE = "patient/visit_date"
COL_CLINIC = "clinic/clinic_name"
COL_SUBMISSION_TIME = "_submission_time"

# =========================
# 3) FUNCTIONS
# =========================

def clean_name(value):
    if pd.isna(value):
        return ""
    value = str(value).strip()
    value = re.sub(r"\s+", " ", value)
    value = value.replace("ـ", "")
    replacements = {
        "أ": "ا", "إ": "ا", "آ": "ا",
        "ى": "ي", "ة": "ه",
        "ؤ": "و", "ئ": "ي",
    }
    for old, new in replacements.items():
        value = value.replace(old, new)
    value = re.sub(r"[^\u0600-\u06FFa-zA-Z0-9\s]", "", value)
    return value.strip()

def clean_id(value):
    if pd.isna(value):
        return ""
    value = str(value).strip()
    value = value.replace(".0", "")
    value = re.sub(r"\D", "", value)
    return value

def clean_gender(value):
    if pd.isna(value):
        return ""
    value = str(value).strip().lower()
    male_values = ["male", "m", "ذكر", "رجل", "boy", "man"]
    female_values = ["female", "f", "انثى", "أنثى", "امرأة", "امراة", "girl", "woman"]
    if value in male_values:
        return "Male"
    if value in female_values:
        return "Female"
    return value

def clean_birth_year(value):
    if pd.isna(value):
        return pd.NA
    value = str(value).strip().replace(".0", "")
    match = re.search(r"\d{4}", value)
    if not match:
        return pd.NA
    year = int(match.group())
    if year < 1900 or year > CURRENT_YEAR:
        return pd.NA
    return year

def most_common(series):
    values = [x for x in series if pd.notna(x) and str(x).strip() != ""]
    if not values:
        return ""
    return Counter(values).most_common(1)[0][0]

def remove_timezone(df):
    for col in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[col]):
            try:
                df[col] = df[col].dt.tz_localize(None)
            except Exception:
                pass
    return df

# =========================
# 4) PULL FROM KOBO API
# =========================

all_records = []
url = DATA_URL

print("جاري جلب بيانات العيون من KoboToolbox...")
while url:
    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        print(response.text)
        raise SystemExit("API Error")
    data = response.json()
    all_records.extend(data.get("results", []))
    url = data.get("next")

df = pd.json_normalize(all_records)
print("إجمالي السجلات المسحوبة:", len(df))

# =========================
# 5) FILTER DATE & CLINIC
# =========================

if COL_VISIT_DATE in df.columns:
    df[COL_VISIT_DATE] = pd.to_datetime(df[COL_VISIT_DATE], errors="coerce").dt.date

if START_DATE:
    start = pd.to_datetime(START_DATE).date()
    df = df[df[COL_VISIT_DATE] >= start]

if END_DATE:
    end = pd.to_datetime(END_DATE).date()
    df = df[df[COL_VISIT_DATE] <= end]

if CLINIC_FILTER and COL_CLINIC in df.columns:
    df = df[df[COL_CLINIC] == CLINIC_FILTER]

df = df.copy()
print("إجمالي السجلات بعد الفلترة:", len(df))

if len(df) == 0:
    raise SystemExit("لا توجد بيانات تطابق الفلاتر المدخلة.")

df["source_row"] = range(1, len(df) + 1)

# =========================
# 6) STANDARDIZATION
# =========================

df["clean_id"] = df[COL_ID].apply(clean_id)
df["clean_name"] = df[COL_NAME].apply(clean_name)
df["clean_gender"] = df[COL_GENDER].apply(clean_gender)
df["clean_birth_year"] = df[COL_BIRTH_YEAR].apply(clean_birth_year)

# =========================
# 7) MASTER IDENTITY (CONFLICTS)
# =========================

valid_id_df = df[df["clean_id"] != ""].copy()

master = (
    valid_id_df
    .groupby("clean_id")
    .agg(
        visits_count=("clean_id", "count"),
        master_name=("clean_name", most_common),
        master_gender=("clean_gender", most_common),
        master_birth_year=("clean_birth_year", most_common),
        unique_names=("clean_name", lambda x: sorted(set([v for v in x if v != ""]))),
        unique_genders=("clean_gender", lambda x: sorted(set([v for v in x if v != ""]))),
        unique_birth_years=("clean_birth_year", lambda x: sorted(set([v for v in x if pd.notna(v)]))),
    )
    .reset_index()
)

master["problem_same_id_different_name"] = master["unique_names"].apply(lambda x: len(x) > 1)
master["problem_same_id_different_gender"] = master["unique_genders"].apply(lambda x: len(x) > 1)
master["problem_same_id_different_birth_year"] = master["unique_birth_years"].apply(lambda x: len(x) > 1)

master["has_identity_problem"] = (
    master["problem_same_id_different_name"] |
    master["problem_same_id_different_gender"] |
    master["problem_same_id_different_birth_year"]
)

df = df.merge(
    master[
        [
            "clean_id", "visits_count", "master_name", "master_gender", "master_birth_year",
            "problem_same_id_different_name", "problem_same_id_different_gender",
            "problem_same_id_different_birth_year", "has_identity_problem"
        ]
    ],
    on="clean_id",
    how="left"
)

# =========================
# 8) BASIC PROBLEMS
# =========================

df["problem_missing_id"] = df["clean_id"].eq("")
df["problem_missing_name"] = df["clean_name"].eq("")
df["problem_missing_gender"] = df["clean_gender"].eq("")
df["problem_missing_or_invalid_birth_year"] = df["clean_birth_year"].isna()

df["problem_id_too_short"] = df["clean_id"].apply(lambda x: x != "" and len(x) < 6)

df["problem_fake_id"] = df["clean_id"].isin([
    "0", "00", "000", "0000", "00000", "000000", "000000000",
    "111111111", "123456789", "999999999", "123456", "123123"
])

# =========================
# 9) FLAGS & RISK LEVELS
# =========================

problem_columns = [
    "problem_missing_id",
    "problem_missing_name",
    "problem_missing_gender",
    "problem_missing_or_invalid_birth_year",
    "problem_id_too_short",
    "problem_fake_id",
    "problem_same_id_different_name",
    "problem_same_id_different_gender",
    "problem_same_id_different_birth_year",
]

labels = {
    "problem_missing_id": "MISSING_ID",
    "problem_missing_name": "MISSING_NAME",
    "problem_missing_gender": "MISSING_GENDER",
    "problem_missing_or_invalid_birth_year": "MISSING_OR_INVALID_BIRTH_YEAR",
    "problem_id_too_short": "ID_TOO_SHORT",
    "problem_fake_id": "FAKE_OR_TEST_ID",
    "problem_same_id_different_name": "SAME_ID_DIFFERENT_NAME",
    "problem_same_id_different_gender": "SAME_ID_DIFFERENT_GENDER",
    "problem_same_id_different_birth_year": "SAME_ID_DIFFERENT_BIRTH_YEAR",
}

def build_flags(row):
    return " | ".join([labels[col] for col in problem_columns if row.get(col) is True])

df["problem_flags"] = df.apply(build_flags, axis=1)

def risk_level(flags):
    if flags == "":
        return "Clean"

    critical_flags = [
        "MISSING_ID",
        "FAKE_OR_TEST_ID",
        "SAME_ID_DIFFERENT_GENDER",
        "SAME_ID_DIFFERENT_BIRTH_YEAR",
    ]

    if any(flag in flags for flag in critical_flags):
        return "Critical"
    return "Warning"

df["risk_level"] = df["problem_flags"].apply(risk_level)

# =========================
# 10) ISSUE SHEETS PREP
# =========================

base_cols = [
    "_id", "source_row", COL_VISIT_DATE, COL_CLINIC, COL_ID, COL_NAME, COL_GENDER, COL_BIRTH_YEAR,
    "clean_id", "clean_name", "clean_gender", "clean_birth_year",
    "master_name", "master_gender", "master_birth_year",
    "problem_flags", "risk_level"
]

base_cols = [c for c in base_cols if c in df.columns]

def issue_sheet(problem_col, issue_name, explanation):
    temp = df[df[problem_col].fillna(False)].copy()
    temp.insert(0, "Issue_Name", issue_name)
    temp.insert(1, "Explanation_Arabic", explanation)
    cols = ["Issue_Name", "Explanation_Arabic"] + base_cols
    cols = [c for c in cols if c in temp.columns]
    return temp[cols]

sheet_missing_id = issue_sheet("problem_missing_id", "MISSING_ID", "رقم هوية المريض مفقود أو فارغ.")
sheet_missing_name = issue_sheet("problem_missing_name", "MISSING_NAME", "اسم المريض مفقود أو فارغ.")
sheet_missing_gender = issue_sheet("problem_missing_gender", "MISSING_GENDER", "جنس المريض مفقود أو غير محدد.")
sheet_missing_birth_year = issue_sheet("problem_missing_or_invalid_birth_year", "MISSING_OR_INVALID_BIRTH_YEAR", "سنة الميلاد مفقودة أو خارج النطاق المنطقي.")
sheet_id_too_short = issue_sheet("problem_id_too_short", "ID_TOO_SHORT", "رقم الهوية المدخل قصير جداً (أقل من 6 أرقام).")
sheet_fake_id = issue_sheet("problem_fake_id", "FAKE_OR_TEST_ID", "رقم الهوية المدخل يبدو وهمياً أو مخصصاً للاختبار.")
sheet_same_id_name = issue_sheet("problem_same_id_different_name", "SAME_ID_DIFFERENT_NAME", "رقم الهوية مرتبط بأكثر من اسم مختلف في السجلات.")
sheet_same_id_gender = issue_sheet("problem_same_id_different_gender", "SAME_ID_DIFFERENT_GENDER", "رقم الهوية مرتبط بأكثر من جنس مختلف في السجلات.")
sheet_same_id_birth_year = issue_sheet("problem_same_id_different_birth_year", "SAME_ID_DIFFERENT_BIRTH_YEAR", "رقم الهوية مرتبط بأكثر من سنة ميلاد مختلفة في السجلات.")

# =========================
# 11) SUMMARY
# =========================

summary_rows = [
    ["Total records", len(df)],
    ["Unique IDs", df["clean_id"].nunique()],
    ["Clean records", int((df["risk_level"] == "Clean").sum())],
    ["Warning records", int((df["risk_level"] == "Warning").sum())],
    ["Critical records", int((df["risk_level"] == "Critical").sum())],
]

for col in problem_columns:
    summary_rows.append([labels[col], int(df[col].fillna(False).sum())])

summary = pd.DataFrame(summary_rows, columns=["Metric", "Count"])

identity_summary = master[master["has_identity_problem"]].copy()
records_with_problems = df[df["problem_flags"] != ""].copy()

# =========================
# 12) EXPORT TO EXCEL
# =========================

tables = [
    df, records_with_problems, identity_summary,
    sheet_missing_id, sheet_missing_name, sheet_missing_gender,
    sheet_missing_birth_year, sheet_id_too_short, sheet_fake_id,
    sheet_same_id_name, sheet_same_id_gender, sheet_same_id_birth_year
]

for i, table in enumerate(tables):
    tables[i] = remove_timezone(table)

with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    summary.to_excel(writer, sheet_name="00_Summary", index=False)
    identity_summary.to_excel(writer, sheet_name="01_ID_Conflicts_Summary", index=False)
    sheet_missing_id.to_excel(writer, sheet_name="02_Missing_ID", index=False)
    sheet_missing_name.to_excel(writer, sheet_name="03_Missing_Name", index=False)
    sheet_missing_gender.to_excel(writer, sheet_name="04_Missing_Gender", index=False)
    sheet_missing_birth_year.to_excel(writer, sheet_name="05_BirthYear_Issues", index=False)
    sheet_id_too_short.to_excel(writer, sheet_name="06_ID_Too_Short", index=False)
    sheet_fake_id.to_excel(writer, sheet_name="07_Fake_ID", index=False)
    sheet_same_id_name.to_excel(writer, sheet_name="08_Same_ID_Name", index=False)
    sheet_same_id_gender.to_excel(writer, sheet_name="09_Same_ID_Gender", index=False)
    sheet_same_id_birth_year.to_excel(writer, sheet_name="10_Same_ID_BirthYear", index=False)
    records_with_problems.to_excel(writer, sheet_name="11_All_Problems", index=False)
    df.to_excel(writer, sheet_name="12_All_Checked", index=False)

print("✅ تمت عملية التدقيق لبيانات العيون بنجاح!")
print(f"تم حفظ الملف باسم: {OUTPUT_FILE}")
print("\n--- الإحصائيات العامة ---")
print(summary.to_string(index=False))

جاري جلب بيانات العيون من KoboToolbox...
إجمالي السجلات المسحوبة: 31484
إجمالي السجلات بعد الفلترة: 3354
✅ تمت عملية التدقيق لبيانات العيون بنجاح!
تم حفظ الملف باسم: Eye_Clinic_Data_Quality_Audit.xlsx

--- الإحصائيات العامة ---
                       Metric  Count
                Total records   3354
                   Unique IDs   3015
                Clean records   3334
              Warning records     18
             Critical records      2
                   MISSING_ID      0
                 MISSING_NAME      0
               MISSING_GENDER      0
MISSING_OR_INVALID_BIRTH_YEAR      0
                 ID_TOO_SHORT      0
              FAKE_OR_TEST_ID      0
       SAME_ID_DIFFERENT_NAME     18
     SAME_ID_DIFFERENT_GENDER      2
 SAME_ID_DIFFERENT_BIRTH_YEAR      0
